In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset
from transformers import pipeline

In [2]:
dataset = load_dataset("imdb")

In [3]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(500))
test_dataset = dataset["test"].shuffle(seed=42).select(range(100))

In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [5]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

In [6]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 100/100 [00:00<00:00, 1318.49 examples/s]


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={0:"NEGATIVE",1:"POSITIVE"},
    label2id={"NEGATIVE":0,"POSITIVE":1}
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4981.95it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    logging_steps=10
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [10]:
trainer.train()

d:\Entri App\Coding Files\hugging_face\sample_2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.687037
20,0.653473
30,0.532351
40,0.399837
50,0.266627
60,0.286779


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.64s/it]


TrainOutput(global_step=64, training_loss=0.4525166032835841, metrics={'train_runtime': 1131.3813, 'train_samples_per_second': 0.884, 'train_steps_per_second': 0.057, 'total_flos': 132467398656000.0, 'train_loss': 0.4525166032835841, 'epoch': 2.0})

In [11]:
classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)


In [12]:
print(classifier("The movie was amazing and I loved it!"))
print(classifier("This movie was terrible"))

[{'label': 'POSITIVE', 'score': 0.863349199295044}]
[{'label': 'NEGATIVE', 'score': 0.8250967860221863}]
